# PEFT Basics: Prompt Tuning Qwen3.5-2B-Base

Практический notebook по Parameter-Efficient Fine-Tuning (PEFT) на `Qwen/Qwen3.5-2B-Base`.

Используется **Prompt Tuning**: веса базовой causal language model остаются замороженными, а обучаются только virtual prompt embeddings. SST-2 представляется как generative classification с verbalizers `negative` и `positive`.

`trl.SFTTrainer` отвечает за prompt-completion preprocessing и completion-only loss. Основная оценка использует полную conditional log-likelihood допустимых labels, а свободная генерация отдельно проверяет no-thinking output protocol.

## Теоретическая часть

### Что такое fine-tuning

Предобученная языковая модель уже умеет работать с языком и хранит большое количество общих знаний, но это не означает, что она оптимально решает конкретную прикладную задачу.

**Fine-tuning** — это продолжение обучения pretrained model на специализированных данных. Он позволяет адаптировать модель к определённому домену, формату ответа, стилю, классификации, извлечению информации или другой downstream-задаче без обучения модели с нуля.

### Full fine-tuning

При классическом **full fine-tuning** практически все параметры модели остаются trainable (изменяются).

Градиенты проходят через всю сеть, optimizer обновляет все или почти все веса, а для каждой новой задачи обычно получается отдельная полностью дообученная копия модели.

Упрощённо:

```text
Pretrained model
      │
      ▼
Все параметры trainable
      │
      ▼
Training
      │
      ▼
Полная новая версия модели
```

### Почему full fine-tuning дорогой

Для моделей с миллиардами параметров стоимость обучения определяется не только размером файла с весами.

Во время обучения в GPU memory находятся:

- параметры модели;
- gradients;
- optimizer states;
- activations;
- временные tensors и buffers.

Например, 2 миллиарда параметров в BF16 занимают примерно 4 GB только для хранения самих весов. Реальный training footprint значительно больше.

Кроме того, если одну модель адаптировать к десяти задачам через full fine-tuning, приходится хранить десять почти полных копий модели.

### Что такое PEFT и что он экономит

**PEFT — Parameter-Efficient Fine-Tuning** — семейство методов, которые адаптируют pretrained model, обучая только небольшую часть параметров. Большая часть модели остаётся frozen, а task-specific состояние хранится в компактном adapter.

PEFT уменьшает число trainable parameters, память для gradients и optimizer states, размер checkpoints и стоимость хранения нескольких специализированных версий модели. При этом frozen base model всё равно участвует в forward/backward pass и обычно остаётся в GPU memory.

PEFT как идея существовал до библиотеки Hugging Face. Библиотека `peft` предоставляет единый API для Prompt Tuning, Prefix Tuning, LoRA и других методов и интегрируется с `transformers` и `trl`.

```text
AutoModelForCausalLM
        ↓
PEFT configuration
        ↓
get_peft_model(...)
        ↓
SFTTrainer
        ↓
compact adapter checkpoint
```

### Основные семейства PEFT

| Семейство | Что обучается | Примеры |
|---|---|---|
| Soft prompting | Виртуальные trainable embeddings | Prompt Tuning, Prefix Tuning, P-Tuning |
| Low-rank adaptation | Небольшие low-rank matrices | LoRA, AdaLoRA |
| Adapter methods | Компактные дополнительные transformations | IA3 и другие adapters |
| Selective tuning | Только выбранные существующие параметры | Trainable tokens, LayerNorm tuning |

LoRA — только один из PEFT-методов, хотя сегодня он является наиболее распространённым.

### Краткая хронология

| Год | Метод | Основная идея |
|---:|---|---|
| 2019 | Adapters | Небольшие trainable modules внутри frozen Transformer |
| 2021 | Prefix Tuning | Trainable continuous prefixes |
| 2021 | Prompt Tuning | Trainable soft prompt embeddings |
| 2021 | P-Tuning | Continuous prompts с prompt encoder |
| 2021 | LoRA | Low-rank decomposition обновления весов |
| 2023 | QLoRA | LoRA поверх 4-bit quantized base model |

В этом notebook используется **Prompt Tuning**, потому что на нём особенно наглядно видно базовый принцип PEFT.

### Hard prompt, soft prompt и Prompt Tuning

Обычная текстовая инструкция — это **hard prompt**. Она состоит из реальных tokens словаря и задаётся человеком.

**Soft prompt** состоит из trainable vectors в embedding space. Эти vectors не обязаны соответствовать словам и изменяются через backpropagation.

Prompt Tuning добавляет virtual tokens перед обычными input embeddings:

```text
p1  p2  ...  pk  x1  x2  ...  xn
└─ trainable ─┘  └── input ──┘

base model → frozen
```

Во время training optimizer изменяет только `p1 ... pk`.

### Generative classification и verbalizers

В generative classification каждому классу сопоставляется текстовая строка — **verbalizer**:

```text
0 → " negative"
1 → " positive"
```

Модель не использует отдельную classification head. Она оценивает продолжения через исходную LM head. Для кандидата $y_c=(y_{c,1},\ldots,y_{c,m})$ score определяется полной conditional log-likelihood:

$$
s_c(x)=\sum_{t=1}^{m}\log p(y_{c,t}\mid x,y_{c,<t}).
$$

Предсказанием становится кандидат с максимальным $s_c(x)$. Такой способ поддерживает многотокенные labels и не требует запуска autoregressive reasoning. Однотокенный forced-choice по next-token logits является только частным случаем этой формулы.

### No-thinking output protocol

Для фиксированной sentiment classification reasoning не является частью задачи. Prompt явно завершает пустой блок размышления:

```text
<think>

</think>

```

После него модель должна вернуть только `negative` или `positive`. Это не constrained decoding: свободная генерация всё ещё может нарушить контракт, и notebook измеряет такие случаи через Valid Output Rate и Thinking Output Rate. Во время обучения loss считается только по completion, поэтому `<think>`-токены и текст инструкции не являются targets.

### Full fine-tuning и Prompt Tuning визуально

```text
FULL FINE-TUNING

Task A ──► [ entire model A ]
Task B ──► [ entire model B ]
Task C ──► [ entire model C ]

PROMPT TUNING

Prompt A ─┐
Prompt B ─┼──► [ one frozen base model ]
Prompt C ─┘
```

![Model Tuning vs Prompt Tuning](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/peft/prompt-tuning.png)

Официальная схема Hugging Face показывает, что Prompt Tuning переиспользует одну frozen base model и хранит небольшие task-specific prompts.

### Сколько параметров обучается

Для простого Prompt Tuning число trainable parameters примерно равно:

```text
num_virtual_tokens × hidden_size
```

В notebook используется:

```text
num_virtual_tokens = 16
hidden_size        = 2048
```

Поэтому ожидаемый порядок величины:

```text
16 × 2048 = 32 768 trainable parameters
```

на фоне примерно **2 миллиардов параметров** Qwen3.5-2B-Base.

Фактическое число notebook затем вычисляет автоматически через `requires_grad`.

### Почему Prompt Tuning масштабируется

Размер soft prompt зависит в основном от:

- количества virtual tokens;
- hidden size модели.

Он почти не зависит от общего количества Transformer layers.

Поэтому при переходе от 2B-модели к более крупной модели количество обучаемых параметров растёт намного медленнее, чем размер base model.

### Почему взят Qwen3.5-2B-Base

`Qwen/Qwen3.5-2B-Base` подходит для учебного PEFT notebook по нескольким причинам:

- это современная causal language model с отдельной LM head, поэтому на ней естественно показать generative classification;
- размер уже достаточно велик, чтобы PEFT имел практический смысл;
- модель всё ещё достаточно компактна для локального эксперимента на одной современной GPU;
- Base checkpoint позволяет измерить содержательный zero-shot candidate baseline без случайно инициализированной classification head;
- Prompt Tuning сохраняет исходные weights и создаёт компактный task-specific adapter.

Выбор Base checkpoint означает, что notebook не полагается на скрытое поведение chat template. Формат prompt, no-thinking block и verbalizers задаются явно и одинаково при training и evaluation.

### Prompt Tuning, LoRA и QLoRA

| Метод | Base model | Что обучается | Quantization |
|---|---|---|---|
| Full fine-tuning | trainable | почти все параметры | не обязательна |
| Prompt Tuning | frozen | virtual prompt embeddings | не обязательна |
| LoRA | frozen | low-rank matrices | не обязательна |
| QLoRA | frozen + quantized | LoRA matrices | обычно 4-bit |

Учебная последовательность:

```text
PEFT / Prompt Tuning
        ↓
       LoRA
        ↓
bitsandbytes quantization
        ↓
      QLoRA
```

### Что проверим на практике

В практической части теория проверяется измерениями:

1. сколько параметров содержит Qwen3.5-2B-Base;
2. сколько параметров остаётся trainable после Prompt Tuning;
3. действительно ли base model frozen;
4. насколько Prompt Tuning изменяет полную conditional likelihood labels;
5. насколько candidate prediction согласуется со свободной генерацией;
6. как часто свободная генерация нарушает no-thinking output protocol;
7. насколько confidence scores откалиброваны;
8. воспроизводится ли adapter после сохранения и повторной загрузки;
9. как агрегировать результаты нескольких seed без смешивания dev и test.

### Ограничения Prompt Tuning

Prompt Tuning обучает очень мало параметров, но это не означает, что он всегда лучший вариант.

- Soft prompts не являются человекочитаемыми.
- Качество зависит от initialization, seed, числа virtual tokens и формулировки hard prompt.
- Verbalizers могут иметь tokenizer bias; многотокенные labels требуют оценки всей последовательности.
- Frozen model всё равно участвует в forward/backward pass, поэтому activation memory сохраняется.
- Свободная генерация смешивает качество классификации и соблюдение формата, поэтому она не должна быть единственной метрикой.
- Confidence, полученный из label likelihood, не следует считать откалиброванной вероятностью без отдельной проверки.
- Prompt Tuning обычно имеет меньшую adaptation capacity, чем LoRA, особенно на небольших моделях и сложных задачах.

Поэтому notebook разделяет model selection на dev split и финальную оценку на held-out test split, показывает calibration metrics и сохраняет инфраструктуру для повторных запусков с несколькими seed.

### Источники по теории

- [Hugging Face PEFT — Soft prompts](https://huggingface.co/docs/peft/main/en/conceptual_guides/prompting)
- [Hugging Face PEFT — Methods overview](https://huggingface.co/docs/peft/main/methods/overview)
- [Prompt tuning for causal language modeling](https://huggingface.co/docs/peft/main/task_guides/clm-prompt-tuning)
- [The Power of Scale for Parameter-Efficient Prompt Tuning](https://arxiv.org/abs/2104.08691)
- [GPT Understands, Too](https://arxiv.org/abs/2103.10385)
- [Pattern-Exploiting Training](https://arxiv.org/abs/2001.07676)
- [Parameter-Efficient Transfer Learning for NLP](https://arxiv.org/abs/1902.00751)
- [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)

## Практическая часть

### 1. Импорты и проверка среды

Зависимости устанавливаются в Docker image, а не внутри notebook. `trl.SFTTrainer` используется для prompt-completion preprocessing и обучения, `TorchMetrics` — для classification и calibration metrics, а `pandas` и Plotly — для компактного отображения результатов.

In [1]:
import gc
import json
import math
import os
import sys
import warnings
from contextlib import contextmanager
from importlib.metadata import version as package_version
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import torch
import torch.nn.functional as F
from datasets import DatasetDict, load_dataset
from huggingface_hub import HfApi, create_repo
from IPython.display import display
from peft import (
    AutoPeftModelForCausalLM,
    PromptTuningConfig,
    PromptTuningInit,
    TaskType,
    get_peft_model,
)
from torchmetrics import MetricCollection
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassCalibrationError,
    MulticlassConfusionMatrix,
    MulticlassF1Score,
    MulticlassPrecision,
    MulticlassRecall,
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    set_seed,
)
from trl import SFTConfig, SFTTrainer

for package_name in (
    "torch",
    "transformers",
    "peft",
    "trl",
    "datasets",
    "accelerate",
    "huggingface_hub",
    "plotly",
    "torchmetrics",
):
    print(f"{package_name:16s}: {package_version(package_name)}")

print(f"Python          : {sys.version.split()[0]}")
print(f"CUDA available : {torch.cuda.is_available()}")


torch           : 2.13.0+cu130
transformers    : 5.14.1
peft            : 0.20.0
trl             : 1.10.0
datasets        : 5.0.1
accelerate      : 1.14.0
huggingface_hub : 1.28.0
plotly          : 6.9.0
torchmetrics    : 1.9.0
Python          : 3.10.12
CUDA available : True


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [3]:
def clear_device_memory():
    "Release Python garbage and unused CUDA allocator cache."
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()


### 2. Конфигурация

Notebook поддерживает два режима:

```text
RUN_MODE = "smoke" → 4 000 training examples, один seed
RUN_MODE = "full"  → весь training pool, рекомендуется три seed
```

Режим и seed можно переопределить переменными окружения `PEFT_RUN_MODE` и `PEFT_SEED`. Каждый запуск сохраняется в отдельную папку `seed-<N>`, поэтому результаты нескольких запусков не перезаписывают друг друга.

Official SST-2 validation не используется для early stopping. Из исходного train split выделяется стратифицированный dev split, а official validation становится held-out test split.

In [4]:
DEFAULT_SEED = 42
SEED = int(os.environ.get("PEFT_SEED", DEFAULT_SEED))
RECOMMENDED_FULL_RUN_SEEDS = (13, 42, 73)

MODEL_ID = "Qwen/Qwen3.5-2B-Base"
DATASET_ID = "stanfordnlp/sst2"
MODEL_REVISION = None
DATASET_REVISION = None

TEXT_COLUMN = "sentence"
LABEL_COLUMN = "label"
LABEL_NAMES = {0: "negative", 1: "positive"}
CLASS_LABELS = tuple(LABEL_NAMES.values())
CANDIDATE_TEXTS = tuple(f" {label}" for label in CLASS_LABELS)

RUN_MODE = os.environ.get("PEFT_RUN_MODE", "smoke").strip().lower()
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("PEFT_RUN_MODE must be 'smoke' or 'full'.")

PROMPT_INIT_TEXT = "Classify the sentiment of the movie review as positive or negative."
NUM_VIRTUAL_TOKENS = 16
MAX_LENGTH = 128
DEV_SAMPLES = 256 if RUN_MODE == "smoke" else 872
MAX_TRAIN_SAMPLES = 1_000 if RUN_MODE == "smoke" else None

BASELINE_TEST_SAMPLES = None
FINAL_TEST_SAMPLES = None
GENERATION_BATCH_SIZE = 8
CANDIDATE_BATCH_SIZE = 8
MAX_NEW_TOKENS = 4
CALIBRATION_BINS = 10

ARTIFACT_RELOAD_SAMPLES = 8
RUN_ARTIFACT_RELOAD_TEST = True

TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
NUM_TRAIN_EPOCHS = 3
LEARNING_RATE = 3e-2
LR_SCHEDULER_TYPE = "linear"
WARMUP_FRACTION = 0.05
WEIGHT_DECAY = 0.0
EVAL_STEPS = 100
SAVE_STEPS = EVAL_STEPS
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.0

PUSH_TO_HUB = False
HUB_MODEL_ID = "artyomboyko/qwen3.5-2b-sst2-prompt-tuning"

cwd = Path.cwd().resolve()
if cwd.name == "peft":
    NOTEBOOK_DIR = cwd
elif (cwd / "notebooks" / "finetuning" / "peft").is_dir():
    NOTEBOOK_DIR = (cwd / "notebooks" / "finetuning" / "peft").resolve()
elif Path("/workspace/notebooks/finetuning/peft").is_dir():
    NOTEBOOK_DIR = Path("/workspace/notebooks/finetuning/peft")
else:
    NOTEBOOK_DIR = cwd

OUTPUT_ROOT = NOTEBOOK_DIR / "outputs" / "qwen3.5-2b-sst2-prompt-tuning"
OUTPUT_DIR = OUTPUT_ROOT / f"seed-{SEED}"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
if device.type == "cuda":
    torch.set_float32_matmul_precision("high")

print(f"Run mode             : {RUN_MODE}")
print(f"Seed                 : {SEED}")
print(f"Recommended seeds    : {RECOMMENDED_FULL_RUN_SEEDS}")
print(f"Output directory     : {OUTPUT_DIR}")
print(f"Checkpoint directory : {CHECKPOINT_DIR}")

RESULTS_TABLE = []


Run mode             : smoke
Seed                 : 42
Recommended seeds    : (13, 42, 73)
Output directory     : /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-prompt-tuning/seed-42
Checkpoint directory : /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-prompt-tuning/seed-42/checkpoints


### 3. Загрузка SST-2

Используется Stanford SST-2 с двумя классами: `negative` и `positive`.

Из original train split выделяется стратифицированный dev split для early stopping и выбора checkpoint. Official validation сохраняется как held-out test и не передаётся в `Trainer` во время обучения.

In [5]:
raw_dataset = load_dataset(DATASET_ID, revision=DATASET_REVISION)
train_dev_split = raw_dataset["train"].train_test_split(
    test_size=DEV_SAMPLES,
    seed=SEED,
    shuffle=True,
    stratify_by_column=LABEL_COLUMN,
)


def limit_split(split, max_samples):
    if max_samples is None:
        return split
    return split.shuffle(seed=SEED).select(range(min(max_samples, len(split))))


dataset = DatasetDict(
    train=limit_split(train_dev_split["train"], MAX_TRAIN_SAMPLES),
    dev=train_dev_split["test"],
    test=raw_dataset["validation"],
)

assert len(dataset["dev"]) == DEV_SAMPLES
assert len(dataset["test"]) == len(raw_dataset["validation"])

resolved_dataset_revision = DATASET_REVISION or getattr(
    raw_dataset["train"].info,
    "version",
    None,
)
try:
    resolved_dataset_revision = HfApi().dataset_info(DATASET_ID).sha
except Exception:
    resolved_dataset_revision = (
        str(resolved_dataset_revision) if resolved_dataset_revision else "unavailable"
    )

dataset_fingerprints = {
    split_name: split._fingerprint for split_name, split in dataset.items()
}

print(dataset)
print("Dataset revision:", resolved_dataset_revision)
print("Fingerprints:", dataset_fingerprints)


DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1000
    })
    dev: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 256
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
})
Dataset revision: 8d51e7e4887a4caaa95b3fbebbf53c0490b58bbb
Fingerprints: {'train': '82643c2b02a8ecc5', 'dev': 'b64ab563a85490d6', 'test': 'c1ddc6497ec97f98'}


### 4. Tokenizer и Qwen3.5-2B-Base

Модель загружается без quantization.

Prompt Tuning обучает только virtual prompt embeddings, а исходные weights Qwen3.5 остаются базой для адаптера.

In [6]:
use_bf16 = device.type == "cuda" and torch.cuda.is_bf16_supported()
model_dtype = (
    torch.bfloat16
    if use_bf16
    else torch.float16
    if device.type == "cuda"
    else torch.float32
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=model_dtype,
).to(device)

resolved_model_revision = MODEL_REVISION or getattr(
    base_model.config,
    "_commit_hash",
    None,
)
if not resolved_model_revision:
    try:
        resolved_model_revision = HfApi().model_info(MODEL_ID).sha
    except Exception:
        resolved_model_revision = "unavailable"

print(f"Loaded class   : {type(base_model).__name__}")
print(f"Model dtype    : {next(base_model.parameters()).dtype}")
print(f"Vocabulary     : {len(tokenizer):,}")
print(f"EOS / PAD      : {tokenizer.eos_token!r} / {tokenizer.pad_token!r}")
print(f"Model revision : {resolved_model_revision}")


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loaded class   : Qwen3_5ForCausalLM
Model dtype    : torch.bfloat16
Vocabulary     : 248,077
EOS / PAD      : '<|endoftext|>' / '<|endoftext|>'
Model revision : b1485b2fa6dfa1287294f269f5fb618e03d52d7c


### 5. Параметры base model

До PEFT считаются параметры исходной модели. После создания Prompt Tuning adapter те же функции покажут, какая доля параметров действительно обучается.

In [7]:
def parameter_stats(model):
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    return total, trainable


base_total_params, base_trainable_params = parameter_stats(base_model)
print(f"Total parameters     : {base_total_params:,}")
print(f"Trainable parameters : {base_trainable_params:,}")
print(f"Trainable share      : {base_trainable_params / base_total_params:.6%}")


Total parameters     : 1,881,825,088
Trainable parameters : 1,881,825,088
Trainable share      : 100.000000%


### 6. Формат задачи

SST-2 classification формулируется как standard prompt-completion dataset:

```text
prompt     → instruction + review + empty <think></think> block
completion → " negative" или " positive"
```

`SFTTrainer` добавляет EOS, токенизирует examples и при `completion_only_loss=True` маскирует prompt значением `-100`. В loss участвуют только completion tokens и EOS.

In [8]:
VISIBLE_INSTRUCTION = (
    "Classify the sentiment of this movie review as positive or negative. "
    "Return exactly one label and no explanation."
)
NO_THINK_BLOCK = "<think>\n\n</think>\n\n"


def build_prompt(text):
    return (
        f"{VISIBLE_INSTRUCTION}\n"
        f"Review: {text.strip()}\n"
        f"Sentiment:\n{NO_THINK_BLOCK}"
    )


def format_prompt_completion_batch(examples):
    return {
        "prompt": [build_prompt(text) for text in examples[TEXT_COLUMN]],
        "completion": [CANDIDATE_TEXTS[int(label)] for label in examples[LABEL_COLUMN]],
    }


prompt_completion_dataset = dataset.map(
    format_prompt_completion_batch,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Formatting prompt-completion examples",
)

sample = prompt_completion_dataset["train"][0]
print("Prompt:\n", sample["prompt"])
print("Completion:", repr(sample["completion"]))
print(prompt_completion_dataset)


Formatting prompt-completion examples:   0%|          | 0/1000 [00:00<?, ? examples/s]

Formatting prompt-completion examples:   0%|          | 0/256 [00:00<?, ? examples/s]

Prompt:
 Classify the sentiment of this movie review as positive or negative. Return exactly one label and no explanation.
Review: 's compelling
Sentiment:
<think>

</think>


Completion: ' positive'
DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 1000
    })
    dev: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 256
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 872
    })
})


### 7. Prompt-completion dataset и TRL

Dataset остаётся человекочитаемым и содержит только столбцы `prompt` и `completion`. `SFTTrainer` сам выполняет токенизацию, создаёт completion mask и динамически дополняет batch. Это заменяет ручные `preprocess_batch` и `CausalClassificationCollator`.

### 8. Метрики и функции оценки

Два режима оценки отвечают на разные вопросы:

| Режим | Что измеряет |
|---|---|
| Candidate Likelihood | Какой полный допустимый label вместе с EOS вероятнее |
| Unconstrained Generation | Может ли модель самостоятельно вернуть ровно один label |

Candidate Likelihood является основной classification-оценкой и поддерживает labels любой token-length. Из нормализованных candidate scores рассчитываются Accuracy, Macro F1, Conditional NLL, Brier score и `MulticlassCalibrationError` из TorchMetrics.

Свободная генерация оценивается строгим exact parsing. Строка `positive because...` считается invalid. Дополнительно измеряются Valid Output Rate и Thinking Output Rate. Constrained decoding не используется.

In [9]:
INVALID_LABEL_ID = len(CLASS_LABELS)
NUM_EVAL_CLASSES = INVALID_LABEL_ID + 1

CLASSIFICATION_METRICS = MetricCollection(
    {
        "accuracy": MulticlassAccuracy(NUM_EVAL_CLASSES, average="micro"),
        "precision": MulticlassPrecision(
            NUM_EVAL_CLASSES,
            average=None,
            zero_division=0,
        ),
        "recall": MulticlassRecall(
            NUM_EVAL_CLASSES,
            average=None,
            zero_division=0,
        ),
        "f1": MulticlassF1Score(
            NUM_EVAL_CLASSES,
            average=None,
            zero_division=0,
        ),
        "confusion_matrix": MulticlassConfusionMatrix(NUM_EVAL_CLASSES),
    }
)
CALIBRATION_ERROR = MulticlassCalibrationError(
    num_classes=len(CLASS_LABELS),
    n_bins=CALIBRATION_BINS,
    norm="l1",
)
candidate_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)


@contextmanager
def temporary_padding_side(padding_side):
    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = padding_side
    try:
        yield
    finally:
        tokenizer.padding_side = original_padding_side


def select_eval_split(raw_split, max_samples):
    size = len(raw_split) if max_samples is None else min(max_samples, len(raw_split))
    return raw_split.select(range(size))


def normalize_prediction(text):
    normalized = text.strip().lower()
    return (
        CLASS_LABELS.index(normalized)
        if normalized in CLASS_LABELS
        else INVALID_LABEL_ID
    )


def classification_metrics(predictions, references):
    predictions = torch.as_tensor(predictions, dtype=torch.long)
    references = torch.as_tensor(references, dtype=torch.long)
    metrics = CLASSIFICATION_METRICS.clone()(predictions, references)
    metrics["macro_f1"] = metrics["f1"][: len(CLASS_LABELS)].mean()
    metrics["total"] = references.numel()
    return metrics


def print_classification_summary(title, metrics):
    print(f"\n{title}\n{'-' * len(title)}")
    print(f"Accuracy: {metrics['accuracy']:.2%} | Macro F1: {metrics['macro_f1']:.4f}")
    for index, label in enumerate(CLASS_LABELS):
        print(
            f"{label:8s} precision={metrics['precision'][index]:.4f} "
            f"recall={metrics['recall'][index]:.4f} "
            f"f1={metrics['f1'][index]:.4f}"
        )

    optional_metrics = {
        "conditional_nll": "Conditional NLL",
        "brier_score": "Brier score",
        "ece": "ECE",
        "valid_output_rate": "Valid output rate",
        "thinking_output_rate": "Thinking output rate",
    }
    for metric_name, label in optional_metrics.items():
        if metric_name in metrics:
            print(f"{label}: {metrics[metric_name]:.4f}")

    print("\nConfusion matrix (rows=actual, columns=predicted)")
    print(f"{'':12s}{'negative':>10s}{'positive':>10s}{'invalid':>10s}")
    for label, row in zip(CLASS_LABELS, metrics["confusion_matrix"][:2]):
        print(f"{label:12s}{row[0]:10d}{row[1]:10d}{row[2]:10d}")


def evaluate_generation(model, raw_split, max_samples=None, batch_size=32):
    model.eval()
    eval_split = select_eval_split(raw_split, max_samples)
    model_device = next(model.parameters()).device
    predictions, references, examples = [], [], []
    thinking_outputs = 0

    with temporary_padding_side("left"), torch.inference_mode():
        for start in range(0, len(eval_split), batch_size):
            batch = eval_split[start : start + batch_size]
            inputs = tokenizer(
                [build_prompt(text) for text in batch[TEXT_COLUMN]],
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
            ).to(model_device)

            with warnings.catch_warnings():
                warnings.filterwarnings(
                    "ignore",
                    message="Position ids are not supported for parameter efficient tuning.*",
                    category=UserWarning,
                    module="peft.peft_model",
                )
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            texts = tokenizer.batch_decode(
                outputs[:, inputs["input_ids"].shape[1] :],
                skip_special_tokens=True,
            )
            batch_predictions = [normalize_prediction(text) for text in texts]
            batch_references = [int(label) for label in batch[LABEL_COLUMN]]

            thinking_outputs += sum("<think>" in text.lower() for text in texts)
            predictions.extend(batch_predictions)
            references.extend(batch_references)
            examples.extend(
                (
                    text,
                    None if prediction == INVALID_LABEL_ID else CLASS_LABELS[prediction],
                    CLASS_LABELS[reference],
                )
                for text, prediction, reference in zip(
                    texts,
                    batch_predictions,
                    batch_references,
                )
            )

    metrics = classification_metrics(predictions, references)
    metrics.update(
        valid_output_rate=sum(p != INVALID_LABEL_ID for p in predictions) / len(eval_split),
        thinking_output_rate=thinking_outputs / len(eval_split),
        predictions=predictions,
        references=references,
        examples=examples,
    )
    return metrics


def build_candidate_feature(text, candidate_text):
    candidate_ids = tokenizer(
        candidate_text,
        add_special_tokens=False,
    )["input_ids"] + [tokenizer.eos_token_id]
    max_prompt_length = max(1, MAX_LENGTH - len(candidate_ids))
    prompt_ids = tokenizer(
        build_prompt(text),
        add_special_tokens=False,
        truncation=True,
        max_length=max_prompt_length,
    )["input_ids"]
    return {
        "input_ids": prompt_ids + candidate_ids,
        "attention_mask": [1] * (len(prompt_ids) + len(candidate_ids)),
        "labels": [-100] * len(prompt_ids) + candidate_ids,
    }


def evaluate_candidate_likelihood(model, raw_split, max_samples=None, batch_size=16):
    model.eval()
    eval_split = select_eval_split(raw_split, max_samples)
    model_device = next(model.parameters()).device
    all_scores, references = [], []

    with temporary_padding_side("right"), torch.inference_mode():
        for start in range(0, len(eval_split), batch_size):
            batch = eval_split[start : start + batch_size]
            features = [
                build_candidate_feature(text, candidate)
                for text in batch[TEXT_COLUMN]
                for candidate in CANDIDATE_TEXTS
            ]
            model_inputs = candidate_collator(features)
            labels = model_inputs.pop("labels").to(model_device)
            model_inputs = {
                key: value.to(model_device) for key, value in model_inputs.items()
            }

            logits = model(**model_inputs, use_cache=False).logits
            logits = logits[:, -labels.shape[1] :, :]
            token_nll = F.cross_entropy(
                logits[:, :-1, :].float().transpose(1, 2),
                labels[:, 1:],
                ignore_index=-100,
                reduction="none",
            )
            sequence_scores = -token_nll.sum(dim=-1)
            all_scores.append(
                sequence_scores.view(-1, len(CANDIDATE_TEXTS)).cpu()
            )
            references.extend(int(label) for label in batch[LABEL_COLUMN])

    scores = torch.cat(all_scores)
    probabilities = scores.softmax(dim=-1)
    predictions = scores.argmax(dim=-1)
    references_tensor = torch.tensor(references, dtype=torch.long)
    one_hot_references = F.one_hot(
        references_tensor,
        num_classes=len(CLASS_LABELS),
    ).float()

    metrics = classification_metrics(predictions, references_tensor)
    metrics.update(
        conditional_nll=float(F.cross_entropy(scores, references_tensor)),
        brier_score=float(
            ((probabilities - one_hot_references) ** 2).sum(dim=-1).mean()
        ),
        ece=float(CALIBRATION_ERROR.clone()(probabilities, references_tensor)),
        scores=scores.tolist(),
        probabilities=probabilities.tolist(),
        predictions=predictions.tolist(),
        references=references,
    )
    return metrics


def result_row(variant, candidate_metrics, generation_metrics):
    return {
        "variant": variant,
        "candidate_accuracy": float(candidate_metrics["accuracy"]),
        "candidate_macro_f1": float(candidate_metrics["macro_f1"]),
        "conditional_nll": candidate_metrics["conditional_nll"],
        "brier_score": candidate_metrics["brier_score"],
        "ece": candidate_metrics["ece"],
        "generation_accuracy": float(generation_metrics["accuracy"]),
        "valid_output_rate": generation_metrics["valid_output_rate"],
        "thinking_output_rate": generation_metrics["thinking_output_rate"],
    }


def evaluate_variant(variant, model, raw_split, max_samples=None):
    candidate_metrics = evaluate_candidate_likelihood(
        model,
        raw_split,
        max_samples=max_samples,
        batch_size=CANDIDATE_BATCH_SIZE,
    )
    generation_metrics = evaluate_generation(
        model,
        raw_split,
        max_samples=max_samples,
        batch_size=GENERATION_BATCH_SIZE,
    )

    expected_size = len(select_eval_split(raw_split, max_samples))
    assert candidate_metrics["total"] == expected_size
    assert generation_metrics["total"] == expected_size

    print_classification_summary(
        f"{variant} — candidate likelihood",
        candidate_metrics,
    )
    print_classification_summary(
        f"{variant} — unconstrained generation",
        generation_metrics,
    )
    print("\nUnconstrained generation examples:")
    for generated, prediction, reference in generation_metrics["examples"][:10]:
        print(
            f"generated={generated!r:20s} "
            f"parsed={prediction!r:10s} "
            f"reference={reference}"
        )

    RESULTS_TABLE.append(result_row(variant, candidate_metrics, generation_metrics))
    return candidate_metrics, generation_metrics


### 9. Baseline на held-out test split

До создания Prompt Tuning adapter измеряются две baseline-оценки на fixed held-out test split:

- основная candidate likelihood evaluation;
- дополнительная unconstrained generation evaluation.

Test split не передаётся в `Trainer` и не используется для early stopping. Конфигурация эксперимента фиксируется до просмотра итогового сравнения.

In [10]:
baseline_candidate_metrics, baseline_generation_metrics = evaluate_variant(
    "Исходная модель",
    base_model,
    dataset["test"],
    max_samples=BASELINE_TEST_SAMPLES,
)



Исходная модель — candidate likelihood
--------------------------------------
Accuracy: 90.48% | Macro F1: 0.9046
negative precision=0.8586 recall=0.9650 f1=0.9087
positive precision=0.9616 recall=0.8468 f1=0.9006
Conditional NLL: 0.2412
Brier score: 0.1391
ECE: 0.0309

Confusion matrix (rows=actual, columns=predicted)
              negative  positive   invalid
negative           413        15         0
positive            68       376         0

Исходная модель — unconstrained generation
------------------------------------------
Accuracy: 91.40% | Macro F1: 0.9150
negative precision=0.8865 recall=0.9486 f1=0.9165
positive precision=0.9490 recall=0.8806 f1=0.9136
Valid output rate: 0.9977
Thinking output rate: 0.0000

Confusion matrix (rows=actual, columns=predicted)
              negative  positive   invalid
negative           406        21         1
positive            52       391         1

Unconstrained generation examples:
generated='Positive'           parsed='positive' refere

### 10. Prompt Tuning configuration и PeftModel

`PromptTuningConfig` создаёт text-initialized virtual tokens, после чего `get_peft_model()` связывает adapter с base model. В той же ячейке проверяется, что trainable tensors принадлежат только `prompt_encoder`. Такое объединение уменьшает риск выполнить зависимые ячейки в неправильном порядке.

In [11]:
peft_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init=PromptTuningInit.TEXT,
    num_virtual_tokens=NUM_VIRTUAL_TOKENS,
    prompt_tuning_init_text=PROMPT_INIT_TEXT,
    tokenizer_name_or_path=MODEL_ID,
)
model = get_peft_model(base_model, peft_config)
model.config.use_cache = False

total_params, trainable_params = parameter_stats(model)
trainable_names = [
    name for name, parameter in model.named_parameters() if parameter.requires_grad
]
unexpected_trainable = [
    name for name in trainable_names if "prompt_encoder" not in name
]

model.print_trainable_parameters()
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"Trainable share      : {trainable_params / total_params:.8%}")
print("Trainable tensors    :", *trainable_names, sep="\n - ")

assert not unexpected_trainable, f"Unexpected trainable parameters: {unexpected_trainable}"
assert trainable_params < total_params * 0.001


trainable params: 32,768 || all params: 1,881,857,856 || trainable%: 0.0017
Total parameters     : 1,881,857,856
Trainable parameters : 32,768
Trainable share      : 0.00174126%
Trainable tensors    :
 - prompt_encoder.default.embedding.weight


### 11. SFTConfig, Linear LR scheduler и Early Stopping

`SFTConfig` расширяет `TrainingArguments` параметрами supervised fine-tuning:

- `completion_only_loss=True` — prompt не участвует в loss;
- `loss_type="nll"` — сохраняется обычный causal LM negative log-likelihood;
- `packing=False` — examples не объединяются в общие sequences;
- `max_length=128` — единый предел длины.

Linear scheduler использует 5% optimizer steps для warmup, затем уменьшает learning rate к нулю. Early Stopping контролирует `eval_loss` dev split с `patience=3`. Validation и сохранение checkpoint выполняются каждые `EVAL_STEPS=100` optimizer steps.

In [12]:
training_batches_per_epoch = math.ceil(
    len(prompt_completion_dataset["train"]) / TRAIN_BATCH_SIZE
)
optimizer_steps_per_epoch = math.ceil(
    training_batches_per_epoch / GRADIENT_ACCUMULATION_STEPS
)
estimated_training_steps = math.ceil(
    optimizer_steps_per_epoch * NUM_TRAIN_EPOCHS
)
WARMUP_STEPS = math.ceil(estimated_training_steps * WARMUP_FRACTION)

use_bf16 = device.type == "cuda" and torch.cuda.is_bf16_supported()
use_fp16 = device.type == "cuda" and not use_bf16

training_args = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    loss_type="nll",
    packing=False,
    eval_packing=False,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    logging_strategy="steps",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    bf16=use_bf16,
    fp16=use_fp16,
    tf32=True if device.type == "cuda" else None,
    optim="adamw_torch_fused" if device.type == "cuda" else "adamw_torch",
    dataloader_num_workers=0,
    dataloader_pin_memory=device.type == "cuda",
    report_to="none",
    push_to_hub=False,
)

print(f"Training examples : {len(prompt_completion_dataset['train'])}")
print(f"Eval interval     : {EVAL_STEPS} optimizer steps")
print(f"LR scheduler      : {LR_SCHEDULER_TYPE}")
print(f"Warmup steps      : {WARMUP_STEPS}")
print(f"Checkpoint output : {CHECKPOINT_DIR}")


Training examples : 1000
Eval interval     : 100 optimizer steps
LR scheduler      : linear
Warmup steps      : 10
Checkpoint output : /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-prompt-tuning/seed-42/checkpoints


### 12. SFTTrainer

`SFTTrainer` получает уже созданный `PeftModel`, поэтому момент применения Prompt Tuning остаётся явным. Trainer подготавливает prompt-completion dataset, а optimizer обновляет только parameters с `requires_grad=True`. После подготовки проверяется наличие supervised completion tokens.

In [13]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=prompt_completion_dataset["train"],
    eval_dataset=prompt_completion_dataset["dev"],
    processing_class=tokenizer,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )
    ],
)

supervised_tokens = sum(
    label != -100 for label in trainer.train_dataset[0]["labels"]
)
print("Prepared columns  :", trainer.train_dataset.column_names)
print("Supervised tokens :", supervised_tokens)
assert supervised_tokens > 0


Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/256 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/256 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/256 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/256 [00:00<?, ? examples/s]

Prepared columns  : ['prompt', 'completion', 'input_ids', 'labels']
Supervised tokens : 2


### 13. Обучение

Для продолжения прерванного запуска можно использовать:

```python
trainer.train(resume_from_checkpoint=True)
```

> При запуске Trainer может появиться информационное сообщение о синхронизации PAD/BOS/EOS tokens между tokenizer и model config. В этом notebook `pad_token` при необходимости переиспользует существующий EOS token. Padding positions исключаются из supervised loss через `-100`, поэтому такое сообщение не означает ошибку training setup.

In [14]:
train_result = trainer.train()
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

training_performance = dict(train_result.metrics)
training_history = list(trainer.state.log_history)
print("\nTraining performance:")
for key in (
    "train_runtime",
    "train_samples_per_second",
    "train_steps_per_second",
    "train_loss",
):
    if key in training_performance:
        print(f"{key:28s}: {training_performance[key]}")


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 248044}.


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
100,0.084711,0.094402,0.058869,0.966797,73945.000000
189,0.052010,0.083325,0.068290,0.966797,139326.000000


***** train metrics *****
  epoch                    =        3.0
  total_flos               =  1464084GF
  train_loss               =     0.2026
  train_runtime            = 0:04:37.62
  train_samples_per_second =     10.806
  train_steps_per_second   =      0.681

Training performance:
train_runtime               : 277.6238
train_samples_per_second    : 10.806
train_steps_per_second      : 0.681
train_loss                  : 0.20264633086623338


### 14. Evaluation loss

После training лучший checkpoint повторно оценивается на dev split. Test split не передаётся в `SFTTrainer` и используется только в последующей Candidate Likelihood и unconstrained generation evaluation.

In [15]:
trainer.remove_callback(EarlyStoppingCallback)
dev_metrics = trainer.evaluate(metric_key_prefix="dev")
trainer.log_metrics("dev", dev_metrics)
trainer.save_metrics("dev", dev_metrics)
dev_metrics


Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
0.052010,0.083325,189,0.068290,0.966797,139326.000000


***** dev metrics *****
  dev_loss                 =   0.0833
  eval_entropy             =   0.0683
  eval_mean_token_accuracy =   0.9668
  eval_num_tokens          = 139326.0


{'dev_loss': 0.08332546055316925,
 'eval_entropy': 0.06828972534276545,
 'eval_mean_token_accuracy': 0.966796875,
 'eval_num_tokens': 139326.0}

### 15. Training dynamics

`SFTTrainer.state.log_history` преобразуется в `pandas.DataFrame`. Train/validation loss и learning rate показываются на двух компактных Plotly-графиках без ручного построения каждой trace.

In [16]:
history = pd.DataFrame(training_history)
loss_history = pd.concat(
    [
        history.reindex(columns=["epoch", "loss"]).dropna(subset=["loss"])
        .rename(columns={"loss": "value"})
        .assign(metric="train loss"),
        history.reindex(columns=["epoch", "eval_loss"]).dropna(subset=["eval_loss"])
        .rename(columns={"eval_loss": "value"})
        .assign(metric="validation loss"),
    ],
    ignore_index=True,
)
display(
    px.line(
        loss_history,
        x="epoch",
        y="value",
        color="metric",
        markers=True,
        title="Training and validation loss",
    )
)

learning_rate_history = history.reindex(columns=["epoch", "learning_rate"]).dropna(
    subset=["learning_rate"]
)
if not learning_rate_history.empty:
    display(
        px.line(
            learning_rate_history,
            x="epoch",
            y="learning_rate",
            title="Learning-rate schedule",
        )
    )


### 16. Accuracy после Prompt Tuning

После training на том же fixed held-out test split повторяются candidate likelihood и unconstrained generation evaluation. Candidate likelihood остаётся основной оценкой, а generation показывает соблюдение no-thinking output protocol.

In [17]:
model = trainer.model
model.config.use_cache = True
final_candidate_metrics, final_generation_metrics = evaluate_variant(
    "Prompt Tuning",
    model,
    dataset["test"],
    max_samples=FINAL_TEST_SAMPLES,
)

print("\nComparison:")
for metric_name, label, formatter in (
    ("accuracy", "Candidate accuracy", ".2%"),
    ("macro_f1", "Candidate Macro F1", ".4f"),
    ("conditional_nll", "Conditional NLL", ".4f"),
):
    before = baseline_candidate_metrics[metric_name]
    after = final_candidate_metrics[metric_name]
    print(f"{label:22s}: {before:{formatter}} → {after:{formatter}}")
print(
    "Generation accuracy   : "
    f"{baseline_generation_metrics['accuracy']:.2%} → "
    f"{final_generation_metrics['accuracy']:.2%}"
)
print(
    "Thinking output rate  : "
    f"{baseline_generation_metrics['thinking_output_rate']:.2%} → "
    f"{final_generation_metrics['thinking_output_rate']:.2%}"
)

artifact_expected_generation = final_generation_metrics["predictions"][
    :ARTIFACT_RELOAD_SAMPLES
]
artifact_expected_candidate = final_candidate_metrics["predictions"][
    :ARTIFACT_RELOAD_SAMPLES
]



Prompt Tuning — candidate likelihood
------------------------------------
Accuracy: 93.23% | Macro F1: 0.9323
negative precision=0.9301 recall=0.9322 f1=0.9312
positive precision=0.9345 recall=0.9324 f1=0.9335
Conditional NLL: 0.1999
Brier score: 0.1065
ECE: 0.0314

Confusion matrix (rows=actual, columns=predicted)
              negative  positive   invalid
negative           399        29         0
positive            30       414         0

Prompt Tuning — unconstrained generation
----------------------------------------
Accuracy: 93.12% | Macro F1: 0.9312
negative precision=0.9340 recall=0.9252 f1=0.9296
positive precision=0.9286 recall=0.9369 f1=0.9327
Valid output rate: 1.0000
Thinking output rate: 0.0000

Confusion matrix (rows=actual, columns=predicted)
              negative  positive   invalid
negative           396        32         0
positive            28       416         0

Unconstrained generation examples:
generated=' positive'          parsed='positive' reference=posi

In [18]:
del trainer, train_result, dev_metrics
clear_device_memory()


### 17. Сохранение PEFT adapter

В `OUTPUT_DIR` сохраняются Prompt Tuning adapter и tokenizer.

Base model не копируется: она продолжает использоваться отдельно.

In [19]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved Prompt Tuning adapter to: {OUTPUT_DIR}")


Saved Prompt Tuning adapter to: /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-prompt-tuning/seed-42


#### Reproducibility metadata

Помимо seed notebook фиксирует revisions модели и dataset repository, fingerprints всех реально использованных splits, prompt-completion protocol и training configuration. Фактическое pinning revisions пока не включается: resolved SHA сохраняется как metadata для последующего воспроизведения.

In [20]:
REPRODUCIBILITY_PATH = OUTPUT_DIR / "reproducibility.json"
EVALUATION_SUMMARY_PATH = OUTPUT_DIR / "evaluation_summary.json"

reproducibility_data = {
    "seed": SEED,
    "recommended_full_run_seeds": RECOMMENDED_FULL_RUN_SEEDS,
    "run_mode": RUN_MODE,
    "model_id": MODEL_ID,
    "model_revision_requested": MODEL_REVISION,
    "model_revision_resolved": resolved_model_revision,
    "dataset_id": DATASET_ID,
    "dataset_revision_requested": DATASET_REVISION,
    "dataset_revision_resolved": resolved_dataset_revision,
    "dataset_fingerprints": dataset_fingerprints,
    "train_examples": len(dataset["train"]),
    "dev_examples": len(dataset["dev"]),
    "test_examples": len(dataset["test"]),
    "trainer": "trl.SFTTrainer",
    "method": "prompt_tuning",
    "task_type": "CAUSAL_LM",
    "num_virtual_tokens": NUM_VIRTUAL_TOKENS,
    "prompt_init_text": PROMPT_INIT_TEXT,
    "visible_instruction": VISIBLE_INSTRUCTION,
    "no_think_block": NO_THINK_BLOCK,
    "candidate_texts": CANDIDATE_TEXTS,
    "candidate_scoring": "sum_label_and_eos_log_likelihood",
    "completion_only_loss": True,
    "loss_type": "nll",
    "packing": False,
    "learning_rate": LEARNING_RATE,
    "lr_scheduler_type": LR_SCHEDULER_TYPE,
    "warmup_fraction": WARMUP_FRACTION,
    "warmup_steps": WARMUP_STEPS,
    "weight_decay": WEIGHT_DECAY,
    "eval_steps": EVAL_STEPS,
    "save_steps": SAVE_STEPS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_threshold": EARLY_STOPPING_THRESHOLD,
}
evaluation_summary = {
    "seed": SEED,
    "run_mode": RUN_MODE,
    "baseline": result_row(
        "Исходная модель",
        baseline_candidate_metrics,
        baseline_generation_metrics,
    ),
    "prompt_tuning": result_row(
        "Prompt Tuning",
        final_candidate_metrics,
        final_generation_metrics,
    ),
}
evaluation_summary["baseline"].pop("variant")
evaluation_summary["prompt_tuning"].pop("variant")

REPRODUCIBILITY_PATH.write_text(
    json.dumps(reproducibility_data, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
EVALUATION_SUMMARY_PATH.write_text(
    json.dumps(evaluation_summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("Saved reproducibility metadata:", REPRODUCIBILITY_PATH)
print("Saved evaluation summary:", EVALUATION_SUMMARY_PATH)


Saved reproducibility metadata: /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-prompt-tuning/seed-42/reproducibility.json
Saved evaluation summary: /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-prompt-tuning/seed-42/evaluation_summary.json


### 18. Размер Prompt Tuning adapter

Размер считается только по adapter files, без Trainer checkpoints.

In [21]:
adapter_files = [
    OUTPUT_DIR / "adapter_model.safetensors",
    OUTPUT_DIR / "adapter_config.json",
]
adapter_size_bytes = sum(path.stat().st_size for path in adapter_files if path.exists())

print(f"Prompt Tuning adapter size: {adapter_size_bytes / 1024**2:.3f} MiB")
print("\nFinal export files:")
for file in sorted(OUTPUT_DIR.iterdir()):
    if file.is_file():
        print(f"{file.name:32s} {file.stat().st_size / 1024:.1f} KiB")


Prompt Tuning adapter size: 0.126 MiB

Final export files:
README.md                        2.8 KiB
adapter_config.json              0.6 KiB
adapter_model.safetensors        128.1 KiB
chat_template.jinja              7.6 KiB
evaluation_summary.json          0.7 KiB
reproducibility.json             1.4 KiB
tokenizer.json                   19521.1 KiB
tokenizer_config.json            1.1 KiB


In [22]:
del model, base_model
clear_device_memory()


### 19. Проверка сохранённого adapter

Сохранённый Prompt Tuning adapter загружается одной командой через `AutoPeftModelForCausalLM`. Проверка выполняется на нескольких fixed test examples и сравнивает unconstrained generation и candidate predictions до сохранения и после reload.

In [23]:
if RUN_ARTIFACT_RELOAD_TEST:
    reloaded_model = AutoPeftModelForCausalLM.from_pretrained(
        OUTPUT_DIR,
        dtype=model_dtype,
    ).to(device).eval()

    reload_generation_metrics = evaluate_generation(
        reloaded_model,
        dataset["test"],
        max_samples=ARTIFACT_RELOAD_SAMPLES,
        batch_size=ARTIFACT_RELOAD_SAMPLES,
    )
    reload_candidate_metrics = evaluate_candidate_likelihood(
        reloaded_model,
        dataset["test"],
        max_samples=ARTIFACT_RELOAD_SAMPLES,
        batch_size=ARTIFACT_RELOAD_SAMPLES,
    )

    assert reload_generation_metrics["predictions"] == artifact_expected_generation
    assert reload_candidate_metrics["predictions"] == artifact_expected_candidate
    print(f"Adapter reload smoke test: PASSED ({ARTIFACT_RELOAD_SAMPLES} examples)")

    del reload_generation_metrics, reload_candidate_metrics, reloaded_model
    clear_device_memory()
else:
    print("RUN_ARTIFACT_RELOAD_TEST=False — reload test skipped.")


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Adapter reload smoke test: PASSED (8 examples)


### 20. Создание Hugging Face Model Card

Model Card создаётся как компактный `README.md` для Prompt Tuning adapter. Она описывает prompt-completion protocol, candidate likelihood как основную оценку, held-out test split и отдельную диагностику свободной генерации.

In [24]:
MODEL_CARD_PATH = OUTPUT_DIR / "README.md"
model_display_name = HUB_MODEL_ID.split("/")[-1]
evaluation_scope = f"held-out SST-2 test split ({final_candidate_metrics['total']} examples)"

card_text = f"""---
base_model: {MODEL_ID}
library_name: peft
pipeline_tag: text-generation
datasets:
- {DATASET_ID}
language:
- en
license: apache-2.0
tags:
- peft
- prompt-tuning
- generative-classification
- qwen3.5
- sentiment-analysis
---

# {model_display_name}

Prompt Tuning adapter for
[`{MODEL_ID}`](https://huggingface.co/{MODEL_ID}), trained on
[`{DATASET_ID}`](https://huggingface.co/datasets/{DATASET_ID})
for generative binary sentiment classification with `trl.SFTTrainer`.

The frozen causal LM scores the textual candidates `negative` and `positive`.
No separate classification head is trained.

## Model Details

| Property | Value |
|---|---|
| Base model | `{MODEL_ID}` |
| Method | Prompt Tuning |
| Task type | `CAUSAL_LM` |
| Labels | `negative`, `positive` |
| Virtual tokens | {NUM_VIRTUAL_TOKENS} |
| Prompt initialization | `{PROMPT_INIT_TEXT}` |
| Trainable parameters | {trainable_params:,} |
| Adapter size | {adapter_size_bytes / 1024**2:.2f} MiB |

## Evaluation

Evaluation scope: **{evaluation_scope}**. Early stopping uses a separate dev split.

| Metric | Base model | Prompt Tuning |
|---|---:|---:|
| Candidate accuracy | {baseline_candidate_metrics["accuracy"]:.2%} | **{final_candidate_metrics["accuracy"]:.2%}** |
| Candidate Macro F1 | {baseline_candidate_metrics["macro_f1"]:.4f} | **{final_candidate_metrics["macro_f1"]:.4f}** |
| Conditional NLL | {baseline_candidate_metrics["conditional_nll"]:.4f} | **{final_candidate_metrics["conditional_nll"]:.4f}** |
| Brier score | {baseline_candidate_metrics["brier_score"]:.4f} | **{final_candidate_metrics["brier_score"]:.4f}** |
| ECE | {baseline_candidate_metrics["ece"]:.4f} | **{final_candidate_metrics["ece"]:.4f}** |
| Unconstrained generation accuracy | {baseline_generation_metrics["accuracy"]:.2%} | **{final_generation_metrics["accuracy"]:.2%}** |
| Valid output rate | {baseline_generation_metrics["valid_output_rate"]:.2%} | **{final_generation_metrics["valid_output_rate"]:.2%}** |
| Thinking output rate | {baseline_generation_metrics["thinking_output_rate"]:.2%} | **{final_generation_metrics["thinking_output_rate"]:.2%}** |

## Usage

```python
import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

ADAPTER_ID = "{HUB_MODEL_ID}"
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoPeftModelForCausalLM.from_pretrained(
    ADAPTER_ID,
    dtype="auto",
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()

review = "a wonderfully acted and moving story"
prompt = (
    "{VISIBLE_INSTRUCTION}\n"
    f"Review: {{review}}\n"
    "Sentiment:\n"
    "<think>\n\n</think>\n\n"
)
candidate_texts = {list(CANDIDATE_TEXTS)!r}
```

For classification, score every complete candidate continuation with teacher forcing,
as documented in the training notebook.

## Limitations

- Designed for English SST-2 sentiment classification.
- Requires the documented prompt and verbalizers.
- Candidate confidence is relative to the provided labels and may require calibration.
- Unconstrained generation can still violate the output protocol.
"""

MODEL_CARD_PATH.write_text(card_text, encoding="utf-8")
print(f"Saved Model Card: {MODEL_CARD_PATH}")


Saved Model Card: /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-prompt-tuning/seed-42/README.md


### 21. Публикация в Hugging Face Hub

Push отключён по умолчанию.

Перед публикацией выполните `hf auth login` и установите `PUSH_TO_HUB=True`.

На Hub отправляются Prompt Tuning adapter, tokenizer, `README.md` и `reproducibility.json`. Trainer checkpoints исключаются.

In [25]:
if PUSH_TO_HUB:
    create_repo(repo_id=HUB_MODEL_ID, repo_type="model", exist_ok=True)
    commit_info = HfApi().upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=HUB_MODEL_ID,
        repo_type="model",
        ignore_patterns=[
            "checkpoints/**",
            "checkpoint-*/**",
            "runs/**",
            "*.pt",
            "*.pth",
        ],
        commit_message="Upload Qwen3.5-2B SST-2 Prompt Tuning adapter and model card",
    )
    print(f"Published: https://huggingface.co/{HUB_MODEL_ID}")
    print(f"Commit: {commit_info.commit_url}")
else:
    print("PUSH_TO_HUB=False — nothing was uploaded.")


PUSH_TO_HUB=False — nothing was uploaded.


## Результаты

### Итоговая сравнительная таблица

In [26]:
assert [row["variant"] for row in RESULTS_TABLE] == [
    "Исходная модель",
    "Prompt Tuning",
]

results = pd.DataFrame(RESULTS_TABLE).set_index("variant")
percentage_columns = [
    "candidate_accuracy",
    "generation_accuracy",
    "valid_output_rate",
    "thinking_output_rate",
]
display(
    results.style.format(
        {column: "{:.2%}" for column in percentage_columns}
        | {
            "candidate_macro_f1": "{:.4f}",
            "conditional_nll": "{:.4f}",
            "brier_score": "{:.4f}",
            "ece": "{:.4f}",
        }
    )
)

,candidate_accuracy,candidate_macro_f1,conditional_nll,brier_score,ece,generation_accuracy,valid_output_rate,thinking_output_rate
variant,,,,,,,,
Исходная модель,90.48%,0.9046,0.2412,0.1391,0.0309,91.40%,99.77%,0.00%
Prompt Tuning,93.23%,0.9323,0.1999,0.1065,0.0314,93.12%,100.00%,0.00%


Применение **Prompt Tuning** улучшило качество исходной модели практически по всем ключевым метрикам. `candidate_accuracy` выросла с **90.48% до 93.23%** (+2.75 п.п.), а `candidate_macro_f1` — с **0.9046 до 0.9323**, что указывает на более стабильное качество классификации по всем классам.

Одновременно снизились `conditional_nll` с **0.2412 до 0.1999** и `brier_score` с **0.1391 до 0.1065**. Это означает, что после PEFT модель не только чаще выбирает правильный ответ, но и формирует более качественное распределение вероятностей. `generation_accuracy` также выросла с **91.40% до 93.12%**, а доля корректно сформированных ответов достигла **100%** против 99.77% у исходной модели.

Единственная метрика без улучшения — `ECE`: **0.0309 → 0.0314**. Разница минимальна и показывает, что повышение точности практически не изменило калибровку модели, хотя небольшое ухудшение уверенности стоит учитывать отдельно.

Таким образом, **Prompt Tuning позволил заметно повысить качество модели без полного дообучения её параметров**. Эксперимент подтверждает практическую ценность PEFT: даже обучение небольшого набора дополнительных параметров способно дать измеримый прирост качества при существенно меньших требованиях к вычислительным ресурсам и хранению обучаемых весов.


### Устойчивость по нескольким seed

Для полного отчёта рекомендуется выполнить notebook с `PEFT_RUN_MODE=full` и `PEFT_SEED=13`, `42`, `73`. Каждый запуск сохраняет `evaluation_summary.json` в собственной папке. Следующая ячейка агрегирует все найденные full-run summaries и показывает mean ± standard deviation.

Smoke run с одним seed проверяет работоспособность pipeline, но не оценивает variance метода.

In [27]:
seed_summaries = [
    json.loads(path.read_text(encoding="utf-8"))
    for path in sorted(OUTPUT_ROOT.glob("seed-*/evaluation_summary.json"))
]
seed_summaries = [row for row in seed_summaries if row.get("run_mode") == "full"]

if len(seed_summaries) >= 2:
    seed_results = pd.json_normalize(seed_summaries).set_index("seed")
    prompt_tuning_columns = seed_results.filter(regex=r"^prompt_tuning\.")
    display(prompt_tuning_columns.agg(["mean", "std"]).T)
else:
    print(f"Need at least two full-run summaries; found {len(seed_summaries)}.")


Need at least two full-run summaries; found 0.


## Источники

- [Prompt tuning for causal language modeling](https://huggingface.co/docs/peft/main/task_guides/clm-prompt-tuning)
- [PEFT Soft prompts](https://huggingface.co/docs/peft/main/en/conceptual_guides/prompting)
- [PEFT AutoPeftModel](https://huggingface.co/docs/peft/main/en/package_reference/auto_class)
- [TRL SFTTrainer](https://huggingface.co/docs/trl/sft_trainer)
- [TRL PEFT integration](https://huggingface.co/docs/trl/en/peft_integration)
- [TRL dataset formats](https://huggingface.co/docs/trl/dataset_formats)
- [Qwen3.5-2B-Base](https://huggingface.co/Qwen/Qwen3.5-2B-Base)
- [Transformers Qwen3.5](https://huggingface.co/docs/transformers/model_doc/qwen3_5)
- [TorchMetrics Calibration Error](https://lightning.ai/docs/torchmetrics/stable/classification/calibration_error.html)
- [Stanford SST-2](https://huggingface.co/datasets/stanfordnlp/sst2)
- [The Power of Scale for Parameter-Efficient Prompt Tuning](https://arxiv.org/abs/2104.08691)
- [Pattern-Exploiting Training](https://arxiv.org/abs/2001.07676)
- [Hugging Face PEFT — LoRA sequence classification notebook](https://github.com/huggingface/peft/blob/main/examples/sequence_classification/LoRA.ipynb)